# VGG19: eksperimen regresi PyTorch

Notebook ini mencakup smoke test GPU, full training terkontrol, checkpoint/reload, dan pembacaan hasil validation tiga skenario. Metrik smoke tidak dianggap hasil penelitian. Full training default nonaktif agar tidak terulang saat membuka laporan.

Framework memakai PyTorch/TorchVision atas arahan pengguna. Split, 11 fitur, posisi, target, mask, dan citra PHASE 1 tetap digunakan. Status data sementara sampai klien mengonfirmasi provenance, fitur identik dengan target berbeda, dan AHI ekstrem.

Tahapan: runtime; verifikasi dataset; arsitektur; smoke test; kontrol full training; hasil validation tersimpan. Jalankan dari kernel baru. Output penelitian tetap lokal dalam artifacts.

In [ ]:
from pathlib import Path
import os
import sys
root = next(path for path in (Path.cwd(), *Path.cwd().parents) if (path / 'src' / 'config.py').is_file())
if str(root) not in sys.path:
    sys.path.insert(0, str(root))
os.environ['MPLBACKEND'] = 'Agg'
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')

import json
import pandas as pd
from IPython.display import Image, Markdown, display
from src.config import ARTIFACTS_DIR, SMOKE, TRAINING
from src.model_utils import EXPERIMENTS, configure_runtime, load_development_data, train_experiment
from src.preprocessing import save_json

RUN_SMOKE_TESTS = True
runtime = configure_runtime()
smoke_root = ARTIFACTS_DIR / 'smoke_tests' / 'VGG19'
smoke_root.mkdir(parents=True, exist_ok=True)
if RUN_SMOKE_TESTS:
    save_json(smoke_root / 'runtime.json', runtime)
display(pd.DataFrame([{key: runtime[key] for key in ['framework', 'torch', 'torchvision', 'cuda_runtime', 'compute_device', 'gpu_name']}]))
print('Batch smoke:', SMOKE.batch_size, '| epoch per skenario:', SMOKE.epochs, '| mixed precision:', TRAINING.mixed_precision)

## 1. Verifikasi dataset tersimpan

Hanya train dan validation NPZ yang dimuat. File test tidak digunakan untuk smoke test, pemilihan model, atau tuning. Manifest bersama diperiksa agar identitas dan komponen salinan tidak tumpang tindih. Checksum NPZ, urutan fitur/posisi, dan keselarasan metadata/target harus sesuai.

Input citra tersimpan dalam NHWC uint8. Dataset PyTorch mengubah sumbu menjadi NCHW, tanpa menambahkan ID, IMF, atau AHI sebagai fitur. Posisi masuk sebagai one-hot terpisah. Target float64 asli tetap tersimpan; batch loss memakai float32.

In [ ]:
data, contract = load_development_data()
assert set(data) == {'train', 'validation'}
display(pd.DataFrame([{'subset': split, 'samples': len(payload['targets']), 'image_shape_NHWC': str(payload['images'].shape), 'pixels': str(payload['images'].dtype), 'position_shape': str(payload['position_one_hot'].shape)} for split, payload in data.items()]))
print('Status data:', contract['metadata']['status'])
print('Fingerprint protokol:', contract['fingerprint'])

## 2. Arsitektur dan protokol

TorchVision VGG19 tanpa Batch Normalization memakai convolutional features saja. Classifier ImageNet dibuang; output backbone 512 channel melewati Global Average Pooling. Fitur digabung dengan one-hot posisi, lalu Linear 128, ReLU, dropout 0,3, Linear 64, ReLU, dropout 0,3, dan output linear satu nilai float32. Head mengikuti ResNet50; dimensi input menyesuaikan backbone.

Tiga skenario memakai head dan split yang sama:

1. Scratch: bobot acak, seluruh backbone trainable.
2. Feature extraction: bobot IMAGENET1K_V1, backbone dibekukan dan eval; hanya regression head dilatih.
3. Fine-tuning: mulai dari checkpoint terbaik feature extraction dengan mode dan fingerprint yang sama; hanya block5 dibuka, yaitu convolution pada indeks features 28, 30, 32, dan 34. Optimizer baru memakai learning rate 1e-5.

Normalisasi di dalam model memakai piksel/255, mean [0.485, 0.456, 0.406], dan std [0.229, 0.224, 0.225] dari bobot TorchVision. Resize/crop fotografi bawaan tidak diterapkan karena feature image sudah berukuran 224 x 224 melalui nearest-neighbor. Tidak ada konversi BGR atau normalisasi ganda.

Implementasi memakai src/model_utils.py yang sama, tanpa split atau fit scaler ulang. Feature image merupakan pseudo-image; nomor IMF sama antar sensor tidak membuktikan sinkronisasi fisiologis.

## 3. Smoke test tiga skenario

Setiap skenario memakai satu batch training dan satu batch validation, masing-masing batch 8, selama satu epoch. Subset dipilih deterministik dari manifest; tidak dipilih berdasarkan hasil. Semua angka metrik yang tersimpan adalah diagnostik pipeline, bukan estimasi performa penelitian.

Runner memakai Adam, MSE, AMP/GradScaler, ReduceLROnPlateau, checkpoint terbaik, early stopping, dan penghentian pada nilai non-finite. Output/loss regresi float32. Optimizer update yang dilewati karena overflow AMP dicatat; smoke test gagal bila tidak ada update.

Checkpoint memuat state_dict, konfigurasi, optimizer, scheduler, AMP scaler, epoch, dan fingerprint data. Reload menggunakan weights_only=True dan arsitektur yang dibangun ulang tanpa mengunduh bobot lagi.

In [ ]:
if RUN_SMOKE_TESTS:
    results = []
    feature_checkpoint = None
    for experiment in EXPERIMENTS:
        result, metrics, checkpoint = train_experiment(
            'VGG19', experiment, data, contract, smoke_root,
            mode='smoke', feature_checkpoint=feature_checkpoint,
        )
        results.append(result)
        if experiment == 'feature_extraction':
            feature_checkpoint = checkpoint
    save_json(smoke_root / 'summary.json', results)
else:
    results = json.loads((smoke_root / 'summary.json').read_text(encoding='utf-8'))
    print('Smoke test tidak diulang; membaca hasil GPU yang tersimpan.')
assert all(result['reload_predictions_consistent'] for result in results)
assert all(result['optimizer_updates'] > 0 for result in results)
assert all(not result['test_accessed'] for result in results)
assert all(result['bn_state_unchanged'] for result in results if result['experiment'] != 'scratch')

## 4. Hasil pemeriksaan aktual

Tabel berikut menampilkan biaya dan integritas smoke test. Peak VRAM mengacu pada memori tensor yang dialokasikan PyTorch selama tahap training/validation, bukan seluruh penggunaan GPU oleh aplikasi lain. Ukuran batch yang sama belum menjamin semua full training bebas OOM.

Metrik MAE/MSE/RMSE/R2 per posisi, prediksi negatif, history, dan tujuh grafik disimpan per skenario. R2 tidak didefinisikan jika sampel kurang dari dua atau target konstan. Residual memakai aktual dikurangi prediksi. Prediksi tidak di-clipping. Bland-Altman hanya deskriptif dan mencatat pengukuran berulang pasien.

In [ ]:
summary = pd.DataFrame([{
    'experiment': result['experiment'], 'device': result['compute_device'],
    'train_samples': result['train_samples'], 'validation_samples': result['validation_samples'],
    'total_parameters': result['parameters']['total'], 'trainable': result['parameters']['trainable'],
    'non_trainable': result['parameters']['non_trainable'],
    'training_seconds': result['training_seconds'],
    'peak_allocated_MiB': result['gpu_memory_training'].get('peak_allocated_bytes', 0) / 1024**2,
    'optimizer_updates': result['optimizer_updates'], 'AMP_skipped': result['amp_skipped_steps'],
    'negative_predictions': result['negative_validation_predictions'],
    'reload_consistent': result['reload_predictions_consistent'],
} for result in results])
if RUN_SMOKE_TESTS:
    summary.to_csv(smoke_root / 'smoke_summary.csv', index=False)
display(summary.round(3))
for result in results:
    path = smoke_root / result['experiment'] / 'figures' / 'loss_curves.png'
    display(Image(filename=str(path), width=700))
    print(result['experiment'], ': diagnostik tersimpan, bukan hasil penelitian.')
display(Markdown('**Hasil aktual:** seluruh skenario menyelesaikan optimizer update, checkpoint dan reload konsisten. VGG19 tidak memakai Batch Normalization; pembekuan backbone dan batas block5 diperiksa terpisah. Test NPZ tidak dibaca. Konfirmasi klien tetap diperlukan sebelum finalisasi penelitian.'))

## 5. Kontrol eksekusi full training

Pengguna mengizinkan PHASE 4 sampai full run VGG19 dalam satu arahan. Smoke test dan QA harus lulus sebelum full training dimulai; persetujuan ulang tidak diperlukan. Flag berikut default False agar training tidak terulang saat membaca hasil tersimpan. Aktifkan hanya untuk eksekusi yang telah diizinkan.

Full training memakai seluruh train/validation, maksimal 50 epoch per skenario dan early stopping patience 10. Fine-tuning full wajib mulai dari checkpoint terbaik feature extraction full, bukan checkpoint smoke. Adam/MSE, batch 8, AMP, seed, dan jadwal learning rate mengikuti konfigurasi arsitektur sebelumnya.

Tidak ada evaluasi test otomatis. PHASE 5 perbandingan model menunggu persetujuan berikutnya.

In [ ]:
RUN_FULL_EXPERIMENTS = False
if RUN_FULL_EXPERIMENTS:
    full_root = ARTIFACTS_DIR / 'experiments' / 'VGG19'
    full_results = []
    full_feature_checkpoint = None
    for experiment in EXPERIMENTS:
        result, metrics, checkpoint = train_experiment(
            'VGG19', experiment, data, contract, full_root,
            mode='full', feature_checkpoint=full_feature_checkpoint,
            full_training_approved=RUN_FULL_EXPERIMENTS,
        )
        full_results.append(result)
        if experiment == 'feature_extraction':
            full_feature_checkpoint = checkpoint
    save_json(full_root / 'summary.json', full_results)
else:
    print('Full training tidak dijalankan ulang pada eksekusi notebook ini.')

## 6. Hasil full training yang tersimpan

Bagian ini membaca hasil tiga eksperimen penuh tanpa melatih ulang. Kriteria pemilihan ditetapkan sebelum training: RMSE validation terkecil, lalu MAE jika seri. Model terbaik hanya dipilih di antara tiga skenario VGG19; perbandingan antararsitektur masih menunggu fase selanjutnya.

Laporkan metrik validation keseluruhan dan per posisi beserta jumlah sampel. Waktu kumulatif fine-tuning memasukkan biaya feature extraction sebelumnya. Checkpoint tiap skenario dipilih berdasarkan val_loss terbaik dan dimuat ulang untuk prediksi float32.

Hasil masih sementara karena provenance dan AHI ekstrem belum dikonfirmasi. Validation dipakai untuk pengembangan/pemilihan, sehingga hasil ini belum merupakan estimasi test atau bukti validasi klinis. Tidak ada test evaluation di notebook ini.

In [ ]:
from src.reporting import summarize_full_experiments
full_root = ARTIFACTS_DIR / 'experiments' / 'VGG19'
if (full_root / 'summary.json').is_file():
    comparison, position_metrics, selection = summarize_full_experiments(full_root)
    display(comparison[['Experiment', 'samples', 'MAE', 'MSE', 'RMSE', 'R2', 'epochs', 'best_epoch', 'negative_predictions']].round(4))
    display(comparison[['Experiment', 'trainable_parameters', 'training_seconds', 'cumulative_training_seconds', 'peak_allocated_MiB', 'optimizer_updates', 'amp_skipped_steps']].round(3))
    display(position_metrics[['Experiment', 'position', 'samples', 'MAE', 'RMSE', 'R2', 'negative_predictions']].round(4))
    for experiment in EXPERIMENTS:
        display(Image(filename=str(full_root / experiment / 'figures' / 'loss_curves.png'), width=700))
    display(Markdown(f"**Pemilihan aktual:** {selection['experiment']} memiliki RMSE validation {selection['RMSE']:.4f} dan MAE {selection['MAE']:.4f}. Pemilihan ini memakai validation; test belum dievaluasi."))
else:
    print('Belum ada hasil full training tersimpan. Smoke test tidak dianggap hasil penelitian.')